In [1]:
import duckdb #library for large sql files

con = duckdb.connect() 
trips = "'../data/raw/yellow_tripdata_2026-04.parquet'"

con.sql(f"SELECT * FROM {trips} LIMIT 5").df()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2026-04-01 00:40:05,2026-04-01 00:52:44,1,2.80,1,N,237,68,1,15.6,4.25,0.5,4.25,0.00,1.0,25.60,2.5,0.0,0.75
1,2,2026-04-01 00:09:19,2026-04-01 00:21:29,1,7.37,1,N,138,75,1,28.2,6.00,0.5,9.03,7.46,1.0,54.19,0.0,2.0,0.00
2,2,2026-04-01 00:15:29,2026-04-01 00:34:14,1,7.66,1,N,138,112,1,31.7,6.00,0.5,5.00,0.00,1.0,46.20,0.0,2.0,0.00
3,1,2026-04-01 00:14:20,2026-04-01 00:27:49,0,7.90,1,N,138,262,1,31.0,10.50,0.5,10.09,7.46,1.0,60.55,2.5,2.0,0.00
4,2,2026-04-01 00:04:53,2026-04-01 00:11:54,1,1.34,1,N,230,234,1,8.6,1.00,0.5,2.87,0.00,1.0,17.22,2.5,0.0,0.75


In [4]:
con.sql(f"SELECT COUNT(*) AS total_of_rows FROM {trips}").df()

,total_of_rows
0,3831240


In [25]:
con.sql(f"""
        DESCRIBE
        SELECT *
        FROM {trips}""").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [33]:
con.sql(f"""
    SELECT COUNT(*) AS trips_outside_april
    FROM {trips}
    WHERE tpep_pickup_datetime < '2026-04-01'
       OR tpep_pickup_datetime >= '2026-05-01'
""").df()

,trips_outside_april
0,12


In [21]:
con.sql(f"""
SELECT
        COUNT(*) FILTER (WHERE trip_distance == 0) AS zero_distance,
        COUNT(*) FILTER (WHERE total_amount < 0 ) AS negative_amount
        FROM {trips}
""").df()

,zero_distance,negative_amount
0,93515,14874


In [34]:
con.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE tpep_dropoff_datetime - tpep_pickup_datetime > INTERVAL '3 hours') AS longer_than_3h,
    COUNT(*) FILTER (WHERE tpep_dropoff_datetime < tpep_pickup_datetime)                    AS ends_before_start
FROM {trips}
""").df()

,longer_than_3h,ends_before_start
0,1251,0


In [44]:
con.sql(f"""
    SELECT 
    COUNT(*) AS clean_trips FROM {trips}
    WHERE tpep_pickup_datetime >= '2026-04-01'
       AND tpep_pickup_datetime < '2026-05-01'
       AND trip_distance > 0
       AND total_amount >= 0
        AND tpep_dropoff_datetime > tpep_pickup_datetime
        AND tpep_dropoff_datetime - tpep_pickup_datetime <= INTERVAL '3 hours'
""").df()

,clean_trips
0,3674591


In [46]:
#Estimated was aroun 2.9 loss but I had 4.1, so I have to check why it happened exactly
con.sql(f"""
SELECT
    COUNT(*) FILTER (WHERE tpep_dropoff_datetime = tpep_pickup_datetime) AS zero_duration,
    COUNT(*) FILTER (WHERE trip_distance < 0)                           AS negative_distance,
    COUNT(*) FILTER (WHERE trip_distance IS NULL
                        OR total_amount IS NULL
                        OR tpep_pickup_datetime IS NULL
                        OR tpep_dropoff_datetime IS NULL)                AS has_nulls
FROM {trips}
""").df()

,zero_duration,negative_distance,has_nulls
0,49511,0,0


In [47]:
# Saved query with all cleaning rules, so we dont have to repeat the WHERE every time
con.sql(f"""
CREATE OR REPLACE VIEW clean_trips AS
SELECT *
FROM {trips}
WHERE tpep_pickup_datetime >= '2026-04-01'
  AND tpep_pickup_datetime <  '2026-05-01'
  AND trip_distance > 0
  AND total_amount >= 0
  AND tpep_dropoff_datetime > tpep_pickup_datetime
  AND tpep_dropoff_datetime - tpep_pickup_datetime <= INTERVAL '3 hours'
""")

CountBIGINT


In [ ]:
#Check that values are right 
con.sql("SELECT COUNT(*) AS clean_trips FROM clean_trips").df()

,clean_trips
0,3674591
